# Encode article text — Kaggle (or Colab) GPU

Produces the article vectors for the **semantic axis** (Q3, "compute your own using
BERT/XLM-RoBERTa") in exactly the shape `retrieval/embeddings.py` already reads, so wiring the
result in is a one-line config change and no code change.

**Model:** `sentence-transformers/paraphrase-multilingual-mpnet-base-v2` — XLM-RoBERTa base,
further trained on paraphrase pairs so cosine similarity is meaningful. That last part is the
whole point: raw multilingual BERT scored **0.4857 AUC on EB-NeRD, worse than random**, because
models trained to fill in blanked-out words produce sentence vectors that all look alike. One
model covers Danish and English, which also removes a confound — until now EB-NeRD used word2vec
and MIND used MiniLM, so "is Danish harder?" was tangled up with "is word2vec worse?".

**Why not locally:** this is inference, a couple of minutes on any GPU. It runs here only because
the weights are ~1.1 GB and the local link to HuggingFace measured 1.5 kB/s.

**The output is much bigger than the input** — ~73 MB for `ebnerd_small`, ~229 MB for
`mind_small`. On a slow link, encode what you like but **retrieve `ebnerd_small` first**: EB-NeRD
is where the semantic axis currently loses to BM25, so that one file decides whether this is
worth adopting. MIND's MiniLM already beats BM25 by 0.069 and is not the problem.

Turn the GPU on first — Kaggle: *Settings -> Accelerator -> **GPU T4 x2***. Colab: *Runtime ->
Change runtime type -> GPU*.

**Not the P100.** Kaggle offers it, but its stock PyTorch has no kernels for Pascal (sm_60), so
every CUDA call fails with `no kernel image is available`. Only one of the two T4s is used.

In [ ]:
!pip -q install "sentence-transformers>=3.0" polars pyarrow
import torch

assert torch.cuda.is_available(), "No GPU. Kaggle: Settings -> Accelerator -> GPU T4 x2."
name = torch.cuda.get_device_name(0)
major, minor = torch.cuda.get_device_capability(0)
supported = torch.cuda.get_arch_list()
print(f"GPU: {name}  (compute capability sm_{major}{minor})")
print("this torch build has kernels for:", " ".join(supported))

# Kaggle still offers the P100, but its stock torch ships no sm_60 kernels, so every CUDA call
# dies with "no kernel image is available" - and only *after* the 1.1 GB model download. Fail
# here instead, with the fix.
assert (major, minor) >= (7, 0), (
    f"{name} is sm_{major}{minor}; this torch supports sm_70 and up. "
    "Switch Settings -> Accelerator to 'GPU T4 x2' and re-run."
)

# fp16 needs Volta or newer to be both correct and fast; gate the .half() on it rather than
# assuming.
FP16 = (major, minor) >= (7, 0)
print("fp16:", FP16)

## 1. Point at the input files

**Kaggle** — upload `scratchpad/encode_inputs/*.parquet` as a Dataset (*+ Add Data -> New
Dataset*), and it appears under `/kaggle/input/<name>/`.

**Colab** — run `from google.colab import files; files.upload()` in a cell first; the files land
in the working directory.

In [ ]:
import glob, os
from pathlib import Path

ON_KAGGLE = Path("/kaggle/input").exists()
OUT_DIR = Path("/kaggle/working") if ON_KAGGLE else Path(".")
inputs = sorted(glob.glob("/kaggle/input/**/*.parquet", recursive=True)) if ON_KAGGLE \
         else sorted(p for p in glob.glob("*.parquet") if not p.endswith("_vectors.parquet"))

print("environment:", "Kaggle" if ON_KAGGLE else "Colab/local")
for p in inputs:
    print(f"  {os.path.getsize(p)/1e6:7.1f} MB  {p}")
assert inputs, "no input parquet found - attach the dataset (Kaggle) or upload (Colab)"

## 2. Encode

Vectors are written **un-normalised**, matching the organisers' provided artifacts —
`load_vectors()` and `pipeline/submit.py` both L2-normalise on the way in, so doing it here would
put that step in two places and make the two arms of the comparison non-identical.

In [ ]:
import time
import numpy as np, polars as pl
from sentence_transformers import SentenceTransformer

MODEL = "sentence-transformers/paraphrase-multilingual-mpnet-base-v2"
BATCH = 256

model = SentenceTransformer(MODEL, device="cuda")
if FP16:
    model.half()      # ~2x faster on Turing+. Skipped on older cards, where it is
                      # either unsupported or no faster (Pascal has no tensor cores).
print("dim:", model.get_sentence_embedding_dimension())

for path in inputs:
    name = Path(path).stem
    df = pl.read_parquet(path)
    assert df.columns == ["article_id", "text"], f"{path}: unexpected columns {df.columns}"

    t0 = time.time()
    vecs = model.encode(df["text"].to_list(), batch_size=BATCH, convert_to_numpy=True,
                        show_progress_bar=True).astype(np.float32)
    dt = time.time() - t0

    assert len(vecs) == df.height, "row count changed during encoding"
    assert np.isfinite(vecs).all(), "non-finite values in the output"
    # A collapsed encoder still emits perfectly finite numbers and then silently scores at
    # random - that is exactly the multilingual-BERT failure. Catch it here, not after a
    # full pipeline run.
    assert vecs.std(axis=0).mean() > 1e-3, "vectors barely vary - encoder likely misconfigured"

    out = OUT_DIR / f"{name}_vectors.parquet"
    pl.DataFrame({"article_id": df["article_id"],
                  "vector": [v.tolist() for v in vecs]}).write_parquet(out, compression="zstd")
    print(f"{name}: {df.height:,} x {vecs.shape[1]} in {dt/60:.1f} min "
          f"({df.height/dt:,.0f}/s) -> {out} ({out.stat().st_size/1e6:.0f} MB)")

## 3. Retrieve the files

**Kaggle** — they are in `/kaggle/working`; grab them from the **Output** tab, or *Save Version*
and pull with `kaggle kernels output <user>/<kernel> -p .`, which is a plain resumable HTTPS
transfer. Prefer that on a bad link: a browser download that dies at 90% starts over.

**Colab** — `files.download()` below, which streams through the browser.

Then locally, in `configs/datasets.yaml`, point both EB-NeRD entries at the file:

```yaml
embeddings: data/interim/xlmr/ebnerd_small_vectors.parquet
```

Keep the `word2vec` line commented alongside — it is the *provided-embeddings* arm of Q3 and the
baseline this must beat **on val** before being adopted. Never pick on test.

In [ ]:
if not ON_KAGGLE:
    from google.colab import files
    for f in sorted(glob.glob("*_vectors.parquet")):
        files.download(f)
else:
    print("Kaggle: use the Output tab, or `kaggle kernels output <user>/<kernel> -p .`")
    for f in sorted(OUT_DIR.glob("*_vectors.parquet")):
        print(f"  {f.stat().st_size/1e6:7.0f} MB  {f}")